# Sentiment Analysis Using LSTM

In [ ]:
import torch
import string
import re
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from torch import nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [ ]:
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
def preprocess(text):
  text = text.lower()
  text = re.sub(r'http\S+|www\S+', '', text)
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'\d+', '', text)
  text = re.sub(r'\s+', ' ', text).strip()
  text = re.sub(r'(\W)\1+', r'\1', text)
  text = re.sub(r'[^\w\s]', '', text)
  return text

In [ ]:
df['review'] = df['review'].apply(preprocess)

In [ ]:
print(df["review"].head(5))

0    one of the other reviewers has mentioned that ...
1    a wonderful little production br br the filmin...
2    i thought this was a wonderful way to spend ti...
3    basically theres a family where a little boy j...
4    petter matteis love in the time of money is a ...
Name: review, dtype: object


In [ ]:
def value_fetcher(my_dict,target_value):
  for idx,val in my_dict.items():
    if int(val) == int(target_value):
      return idx

In [ ]:
vocabulary = set()
for sentence in df['review']:
  vocabulary.update(sentence.split())

pad_token = "<PAD>"
unk_token = "<UNK>"

word_to_idx = {
    pad_token : 0,
    unk_token : 1
}

for word in vocabulary:
  word_to_idx[word] = len(word_to_idx)

max_length =df['review'].astype(str).str.split().str.len().max()

def encode_and_pad(text):
  words = text.split()
  encoded_text = [word_to_idx.get(word,word_to_idx["<UNK>"]) for word in words]
  if len(encoded_text) < max_length:
    encoded_text = encoded_text + [word_to_idx["<PAD>"]] * (max_length - len(encoded_text))
  else:
    encoded_text = encoded_text[:max_length]
  return encoded_text

In [ ]:
train_split , test_split = train_test_split(df,test_size=0.1)
train_split.reset_index(drop=True)
test_split.reset_index(drop=True)

train_split['review'] = train_split['review'].apply(encode_and_pad)
test_split['review'] = test_split['review'].apply(encode_and_pad)

In [ ]:
class CustomDataLoader(Dataset):
  def __init__(self,data):
    self.X = data['review'].values
    self.y = data['sentiment'].values
    self.label_map = {"positive":1,"negative":0}

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):

    text = self.X[index]
    label = self.y[index]

    text_tensor = torch.tensor(text,dtype=torch.long)
    label_tensor = torch.tensor(self.label_map[label],dtype=torch.long)
    return text_tensor , label_tensor

In [ ]:
train_ds = CustomDataLoader(train_split)
test_ds = CustomDataLoader(test_split)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle = True
)

test_loader = DataLoader(
    test_ds,
    batch_size = 32,
    shuffle = False
)

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self,vocab_size,embed_size,hidden_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.embedding = nn.Embedding(vocab_size,embed_size,padding_idx =word_to_idx['<PAD>'])
    self.lstm = nn.LSTM(
        input_size = embed_size,
        hidden_size = hidden_size,
        num_layers = 2,
        batch_first=True,
        dropout = 0.4
    )
    self.dropout = nn.Dropout(0.5)
    self.fc = nn.Linear(hidden_size,2)

  def forward(self,x):
    x = self.embedding(x)
    output, _ = self.lstm(x)
    output = output.mean(dim=1)
    output = self.dropout(output)
    output = self.fc(output)
    return output

In [ ]:
def model_train(model,optimizer,loss_function,dataloader):
  model.train()
  train_loss = 0.0
  correct = 0
  total = 0
  for X,y in dataloader:
    optimizer.zero_grad()
    X = X.to(device)
    y = y.to(device)

    output = model(X)
    loss = loss_function(output,y)

    loss.backward()
    optimizer.step()

    train_loss = train_loss + loss.item()
    correct = correct + (output.argmax(dim=1) == y).sum().item()
    total = total + y.size(0)

  accuracy = correct * 100 / total
  avg_train_loss = train_loss / len(dataloader)

  return accuracy , avg_train_loss

In [ ]:
text,label = next(iter(train_loader))
print(text[0].dtype)

torch.int64


In [ ]:
embed_size = 128
hidden_size = 128
vocab_size = len(word_to_idx)
model = SentimentLSTM(vocab_size,embed_size,hidden_size)
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
epochs = 10

for epoch in range(epochs):
  train_acc , train_loss = model_train(model,optimizer,loss_function,train_loader)

  model.eval()
  with torch.no_grad():
    test_loss = 0.0
    correct = 0
    total = 0

    for X,y in test_loader:
      X = X.to(device)
      y = y.to(device)

      output= model(X)

      loss = loss_function(output,y)

      preds = output.argmax(dim=1)
      correct = correct + (preds == y).sum().item()
      total = total + y.size(0)
      test_loss = test_loss + loss.item()


    test_loss_per_epoch = test_loss / len(test_loader)
    test_acc_per_epoch = correct * 100 / total
  print(f"Epoch : {epoch+1} | Train loss : {train_loss:.4f} | Train accuracy : {train_acc:.2f} | Test loss : {test_loss_per_epoch:.4f} | Test Accuracy : {test_acc_per_epoch:.2f}")

Epoch : 1 | Train loss : 0.5981 | Train accuracy : 68.01 | Test loss : 0.4952 | Test Accuracy : 77.56
Epoch : 2 | Train loss : 0.5874 | Train accuracy : 66.78 | Test loss : 0.5183 | Test Accuracy : 73.76
Epoch : 3 | Train loss : 0.3645 | Train accuracy : 82.82 | Test loss : 0.2588 | Test Accuracy : 89.18
Epoch : 4 | Train loss : 0.1819 | Train accuracy : 93.54 | Test loss : 0.2571 | Test Accuracy : 89.60
Epoch : 5 | Train loss : 0.1145 | Train accuracy : 96.32 | Test loss : 0.2657 | Test Accuracy : 89.62
Epoch : 6 | Train loss : 0.0667 | Train accuracy : 98.08 | Test loss : 0.3546 | Test Accuracy : 89.06
Epoch : 7 | Train loss : 0.0406 | Train accuracy : 98.91 | Test loss : 0.4128 | Test Accuracy : 89.20
Epoch : 8 | Train loss : 0.0269 | Train accuracy : 99.29 | Test loss : 0.4898 | Test Accuracy : 89.12
Epoch : 9 | Train loss : 0.0191 | Train accuracy : 99.53 | Test loss : 0.5462 | Test Accuracy : 88.92
Epoch : 10 | Train loss : 0.0146 | Train accuracy : 99.64 | Test loss : 0.5120 | T

# Next word Prediction using LSTM

In [1]:
import string
import nltk
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from collections import Counter
from nltk.tokenize import word_tokenize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
text = """Rohit Gurunath Sharma was born on April 30, 1987, in Bansod, Nagpur, Maharashtra. His story is one of the most inspiring tales in the history of Indian sports, characterized by early financial struggles and a relentless pursuit of excellence. His father, Gurunath Sharma, worked as a caretaker in a transport firm storehouse, and his mother, Purnima Sharma, was a homemaker. Due to the family's limited income, Rohit spent a significant portion of his childhood living with his grandparents and uncles in Borivali, Mumbai. It was in the narrow lanes of Mumbai that Rohit first began playing cricket, a sport that would eventually define his life and the dreams of millions of Indians. His talent was evident even in his childhood games, where his ability to time the ball set him apart from his peers.
In 1999, Rohit joined a cricket camp with money borrowed from his uncle. It was at this camp that he met Dinesh Lad, a coach who would become the most influential figure in his early cricketing development. Lad was immediately struck by Rohit's natural grace and suggested that he move to Swami Vivekanand International School, where Lad was the coach. However, the school fees were beyond the means of Rohit's family. Recognizing the boy's immense potential, Lad convinced the school director to waive the fees, allowing Rohit to focus entirely on his game. Interestingly, Rohit started his journey as an off-spinner. It was only during a net session that Lad noticed his batting ability and encouraged him to focus on becoming a top-order batsman. This transition was the first major turning point in his career. Rohit quickly became a star in school cricket, scoring a century on his debut in the Giles Shield. His performances at the school level were so dominant that he was soon fast-tracked into the Mumbai age-group teams.
Rohit's rise through the ranks of Mumbai's cricketing circuit was meteoric. He made his List A debut for West Zone against Central Zone in the Deodhar Trophy in March 2005. While he didn't make a massive impact in his first few games, his unbeaten 142 against North Zone in Udaipur later that year brought him into the national spotlight. It was an innings of pure aggression and technical brilliance, proving that he could compete against the best domestic bowlers in the country. Following this, he was selected for the India A tour of Abu Dhabi and Australia. His ability to handle pace and bounce was particularly impressive to the selectors. He made his First-Class debut for India A against New Zealand A in July 2006. Later that year, he finally made his Ranji Trophy debut for Mumbai. In his very first season, he scored a double century against Gujarat, helping Mumbai clinch another title. By the age of 20, Rohit was already being touted as the next big thing in Indian cricket, often compared to the legendary Sachin Tendulkar for his effortless style of play.
In June 2007, Rohit earned his first call-up to the Indian national team for an ODI series in Ireland. Although he didn't get to bat in his debut match, he was soon included in the squad for the inaugural ICC World T20 in South Africa. This tournament served as his global introduction. In a high-stakes match against the hosts South Africa, Rohit scored an unbeaten 50 off 40 balls, helping India post a defendable total and eventually win the game. His composure under pressure was remarkable for a 20-year-old. In the final against Pakistan, he played a crucial cameo of 30 runs off just 16 balls. These small but significant contributions were vital to India winning the trophy, and Rohit returned home as a world champion. Shortly after, he showcased his potential in the ODI format during the CB Series in Australia in 2008. His partnership with Sachin Tendulkar in the first final in Sydney, where he scored 66, was instrumental in India's historic series win on Australian soil.
However, the period between 2009 and 2012 was a challenging era for Rohit. Despite his obvious talent, he struggled with consistency and was often criticized for being too casual with his wicket. This phase was marked by flashes of brilliance followed by long periods of low scores. The lowest point came in 2011, when he was left out of India's squad for the ICC Cricket World Cup. Watching his teammates win the trophy at his home ground in Mumbai from his living room was a devastating experience. He later admitted that this exclusion was a massive wake-up call. He began working harder on his fitness and mental preparation, realizing that talent alone would not be enough to sustain a long international career. He returned to the team with a renewed sense of purpose, but it wasn't until 2013 that his career truly changed forever.
The 2013 ICC Champions Trophy in England was the setting for the most significant tactical decision in modern Indian cricket. Captain MS Dhoni decided to promote Rohit Sharma to the opening slot alongside Shikhar Dhawan. The move was a gamble, as Rohit had primarily been a middle-order batsman. However, the pair clicked instantly. Rohit's ability to see off the new ball and then capitalize on his wide range of strokes made him a lethal opener. India went on to win the Champions Trophy, and Rohit established himself as a permanent fixture at the top of the order. Later that year, in November 2013, he played an innings that cemented his status as a modern great. Against Australia in Bangalore, Rohit scored 209 runs, becoming only the third player in history to score an ODI double century. His innings included a then-record 16 sixes. This was the birth of the Hitman persona, a nickname given to him by fans and commentators for his ability to clear the boundary with ease.
In November 2014, Rohit took his run-scoring to an even more surreal level. Playing against Sri Lanka at Eden Gardens, he smashed 264 runs, the highest individual score in the history of One Day Internationals. This remains one of the most incredible feats in the history of the sport. He reached his hundred in 100 balls and then accelerated at a pace that left the opposition helpless, scoring the next 164 runs in just 73 balls. He became the first and only player to score two double centuries in ODIs, a record he would later extend to three with another double hundred against Sri Lanka in 2017. His dominance in the limited-overs format was now undisputed. During this time, he also became a central figure in the Indian Premier League. Having joined Mumbai Indians in 2011, he was appointed captain in 2013. He led the team to their first title that very year, proving that he was not just a great batsman but also a tactical leader.
The middle years of his career were defined by his role as the backbone of India's white-ball team. In the 2015 World Cup, he scored a vital century in the quarter-final against Bangladesh, helping India reach the semi-finals. While his Test career took longer to flourish due to limited opportunities and injuries, he remained a giant in ODIs and T20Is. By 2017, he was appointed the vice-captain of the limited-overs teams. His partnership with Shikhar Dhawan became one of the most successful opening pairings in history, rivaling the greats like Ganguly and Tendulkar. Off the field, Rohit married Ritika Sajdeh in 2015, and the birth of his daughter Samaira in 2018 added a new dimension to his life, with Rohit often mentioning how fatherhood brought more perspective and calmness to his game.
The 2019 ICC Cricket World Cup in England and Wales was the stage for his most remarkable individual achievement in a single tournament. Rohit became the first player in history to score five centuries in a single edition of the World Cup, surpassing the previous record of four held by Kumar Sangakkara. His hundreds came against South Africa, Pakistan, England, Bangladesh, and Sri Lanka. He finished the tournament as the leading run-scorer with 648 runs at an incredible average of 81. While the team’s journey ended in a heartbreaking semi-final loss to New Zealand, Rohit’s performance established him as the premier opening batsman in the world. His ability to build long innings while maintaining a healthy strike rate became the blueprint for India's success in white-ball cricket.
In late 2019, another significant transformation occurred in Rohit’s career, this time in the longest format of the game. After years of being an intermittent presence in the Test side as a middle-order batter, he was pushed to open the innings against South Africa in Visakhapatnam. The results were instantaneous and spectacular. He scored twin centuries in his first match as a Test opener, becoming only the sixth Indian to achieve the feat. Later in the same series, he struck a majestic double century in Ranchi. This move not only rejuvenated his Test career but also provided India with much-needed stability at the top of the order. His success wasn't limited to home conditions; during the 2021 tour of England, he proved his technique against the swinging ball, scoring a vital century at the Oval and finishing as India's leading run-scorer in the series. By this time, his Test average as an opener had soared to over 50, and he was widely regarded as one of the best all-format players of his generation.
Leadership became the defining theme of Rohit’s final years. In early 2022, he was appointed the full-time captain of the Indian team across all three formats. His captaincy style was characterized by a calm demeanor and a focus on empowering young players. Under his leadership, India adopted a more aggressive and selfless brand of cricket. This was most evident during the 2023 ODI World Cup held in India. Rohit led from the front, taking on the bowlers in the powerplay to set a high tempo for the middle order. India went on a historic run of ten consecutive victories to reach the final. Although the final ended in defeat against Australia, Rohit’s leadership and his tally of 597 runs earned him immense respect from the global cricketing community. He proved that he was willing to sacrifice personal milestones for the sake of the team’s momentum, a quality that became the hallmark of his tenure.
The year 2024 brought the ultimate redemption for the Indian captain. Leading a motivated squad in the ICC T20 World Cup held in the West Indies and the USA, Rohit guided India to an unbeaten championship run. In the final against South Africa in Barbados, India secured a nail-biting victory to end an eleven-year drought for an ICC trophy. For Rohit, it was a poetic full circle, having won the inaugural T20 World Cup as a youngster in 2007. Immediately after the victory, he announced his retirement from T20 internationals, leaving the format as its highest run-scorer and one of its most successful leaders. His legacy in T20 cricket was further cemented by his five IPL titles as captain of the Mumbai Indians, a record that stands as a testament to his tactical brilliance in the shortest format.
The twilight of his career was marked by yet another global triumph. In early 2025, Rohit led India to victory in the ICC Champions Trophy. In the final against New Zealand in Dubai, he played a match-winning knock of 76 runs and was named the Player of the Match. This victory made him the first Indian captain to win back-to-back ICC trophies. Following this success, Rohit announced his retirement from Test cricket on May 7, 2025, choosing to focus his remaining energy on the ODI format. His final chapter concluded in January 2026 during a home ODI series against New Zealand. After nearly two decades of international cricket, he walked off the field for the last time, having hit more international sixes than anyone else in history and having scored more centuries in World Cup matches than any other player. Rohit Sharma’s journey from a young boy in Borivali to the Hitman of world cricket is a story of resilience, transformation, and a relentless pursuit of glory for his country."""

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
tokens = word_tokenize(text.lower())

In [5]:
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

In [6]:
len(vocab)

719

In [7]:
input_sentences = text.split('\n')

In [8]:
def text_to_index(sentence,vocab):
  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab["<UNK>"])
  return numerical_sentence


In [9]:
input_numerical_sentence = []
for sentence in input_sentences:
  input_numerical_sentence.append(text_to_index(word_tokenize(sentence.lower()),vocab))

In [10]:
len(input_numerical_sentence)

13

In [11]:
training_sequence = []
for sentence in input_numerical_sentence:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [12]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max_length = max(len_list)

padded_sequence = []
for sequence in training_sequence:
  padded_sequence.append([0] * (max_length - len(sequence)) + sequence)

padded_sequence = torch.tensor(padded_sequence,dtype=torch.long)
print(padded_sequence.shape)
X = padded_sequence[:,:-1]
y = padded_sequence[:,-1]

torch.Size([2254, 204])


In [13]:
class CustomDataLoader(Dataset):
  def __init__(self,X,y):
    self.X = X
    self.y = y
  def __len__(self):
    return X.shape[0]

  def __getitem__(self,idx):
    return self.X[idx], self.y[idx]


In [14]:
dataset = CustomDataLoader(X,y)
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)


In [15]:
class NextWordPredLSTM(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,100)
    self.lstm = nn.LSTM(100,150,batch_first=True)
    self.fc = nn.Linear(150,vocab_size)

  def forward(self,x):
    x = self.embedding(x)
    intermediate_hidden_state , (final_hidden_state,final_cell_State) = self.lstm(x)
    x = self.fc(final_hidden_state.squeeze(0))
    return x


In [16]:
def model_train(model,optimizer,loss_function,dataloader):
  model.train()
  train_loss = 0.0
  correct = 0
  total = 0
  for X,y in dataloader:
    optimizer.zero_grad()
    X = X.to(device)
    y = y.to(device)

    output = model(X)
    loss = loss_function(output,y)

    loss.backward()
    optimizer.step()

    train_loss = train_loss + loss.item()
    correct = correct + (output.argmax(dim=1) == y).sum().item()
    total = total + y.size(0)

  accuracy = correct * 100 / total
  avg_train_loss = train_loss / len(dataloader)

  return accuracy , avg_train_loss


In [18]:
model = NextWordPredLSTM(len(vocab)).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

epochs = 50
for epoch in range(epochs):
  train_acc , train_loss = model_train(model,optimizer,loss_function,dataloader)

  model.eval()
  with torch.no_grad():
    test_loss = 0.0
    correct = 0
    total = 0

    for X,y in dataloader:
      X = X.to(device)
      y = y.to(device)

      output= model(X)

      loss = loss_function(output,y)

      preds = output.argmax(dim=1)
      correct = correct + (preds == y).sum().item()
      total = total + y.size(0)
      test_loss = test_loss + loss.item()


    test_loss_per_epoch = test_loss / len(dataloader)
    test_acc_per_epoch = correct * 100 / total
  print(f"Epoch : {epoch+1} | Train loss : {train_loss:.4f} | Train accuracy : {train_acc:.2f} | Test loss : {test_loss_per_epoch:.4f} | Test Accuracy : {test_acc_per_epoch:.2f}")

Epoch : 1 | Train loss : 6.1069 | Train accuracy : 5.50 | Test loss : 388.5563 | Test Accuracy : 6.69
Epoch : 2 | Train loss : 5.5845 | Train accuracy : 14.29 | Test loss : 5.5241 | Test Accuracy : 14.29
Epoch : 3 | Train loss : 5.5241 | Train accuracy : 14.29 | Test loss : 5.4358 | Test Accuracy : 21.43
Epoch : 4 | Train loss : 5.4358 | Train accuracy : 21.43 | Test loss : 5.3254 | Test Accuracy : 21.43
Epoch : 5 | Train loss : 5.3254 | Train accuracy : 21.43 | Test loss : 5.1971 | Test Accuracy : 21.43
Epoch : 6 | Train loss : 5.1971 | Train accuracy : 21.43 | Test loss : 5.0539 | Test Accuracy : 21.43
Epoch : 7 | Train loss : 5.0539 | Train accuracy : 21.43 | Test loss : 4.8993 | Test Accuracy : 21.43
Epoch : 8 | Train loss : 4.8993 | Train accuracy : 21.43 | Test loss : 4.7365 | Test Accuracy : 21.43
Epoch : 9 | Train loss : 4.7365 | Train accuracy : 21.43 | Test loss : 4.5684 | Test Accuracy : 21.43
Epoch : 10 | Train loss : 4.5684 | Train accuracy : 21.43 | Test loss : 4.3976 | T

In [32]:
def prediction(model, vocab, text):

  tokenized_text = word_tokenize(text.lower())
  numerical_text = text_to_index(tokenized_text, vocab)

  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)
  with torch.no_grad():
    output = model(padded_text.to(device))

    value, index = torch.max(output, dim=1)

  return text + " " + list(vocab.keys())[index]

In [38]:
num_tokens = 10
input_text = "Rohit Gurunath Sharma was born on"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  input_text = output_text
print(output_text)

Rohit Gurunath Sharma was born on april 30 , 1987 , in bansod , nagpur ,
